In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
churn_dataset = pd.read_csv("/content/drive/MyDrive/customer_churn_dataset for logistic regression task.csv")
churn_dataset.head()

Dataset Overview

In [ ]:

display(churn_dataset.shape)
display(churn_dataset.dtypes)
display(churn_dataset.columns)

Missing Value Analysis

In [ ]:
print("Missing Value in each column")
print(churn_dataset.isnull().sum())
print("Total Missing Values : ",churn_dataset.isnull().sum().sum())
print("Percentage of total missing values : ",(churn_dataset.isnull().sum().sum()/len(churn_dataset))*100)

Duplicate Analysis

In [ ]:
print("No of Duplicated Rows : ",churn_dataset.duplicated().sum())


Stastistical Summary

In [ ]:
# Identify Numerical Columns
display(churn_dataset.select_dtypes(include=['int64','float64']))

# Generate All Statiistical Summary
display(churn_dataset.describe())

# Analyze categorical columns separately
churn_dataset.describe(include='object')

# For Checking the frequency of values
display(churn_dataset["Gender"].value_counts())
display(churn_dataset["ContractType"].value_counts())

Visualize portion

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# USing Count plot to count the categories that how many customers belong to each category
sns.countplot(x="ChurnStatus",data = churn_dataset)
plt.title("Churn Distribution")
plt.show()

# using histogram to shows the distribution that how numerical values distributed
sns.histplot(churn_dataset["MonthlyCharges"],bins=20)
plt.title("Distribution by gender")
plt.show()

# using the box plot to highlight the extreme values to check that is there any outlier
sns.boxplot(y=churn_dataset["MonthlyCharges"])
plt.title("Box PLot of Monthly Charges")
plt.show()

# using correlation heatmap to check that there any two values correlated each other
sns.heatmap(churn_dataset.corr(numeric_only=True),annot=True,cmap="coolwarm")
plt.title("Correlation Heatmap of Numerical Features")
plt.show()

# using count plot with churns to compare churn acrosss groups
sns.countplot(x="ContractType",hue = "ChurnStatus",data = churn_dataset)
plt.title("Churn Distribution Across Contract Type")
plt.show()

Encoding Categorical Values

In [ ]:
# identify categorical columns
churn_dataset.select_dtypes(include = "object").columns

# Check Unique Values
for col in churn_dataset.select_dtypes(include = "object").columns:
  print(f"\ncolumns : {col}")
  print(churn_dataset[col].unique())



Choosing Encoding Methods

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Create LabelEncoder object
label_encoder = LabelEncoder()

# Encode binary columns
churn_dataset["Gender"] = label_encoder.fit_transform(churn_dataset["Gender"])
churn_dataset["ChurnStatus"] = label_encoder.fit_transform(churn_dataset["ChurnStatus"])

# One-Hot Encode multi-category columns
churn_dataset = pd.get_dummies(
    churn_dataset,
    columns=["ContractType", "InternetService", "PaymentMethod"],
    drop_first=True
)

# Display first 5 rows
churn_dataset.head()

In [ ]:
churn_dataset.head()
churn_dataset.info()

Removing Irrelevant Columns

In [ ]:
# This value has no relationship with churn. Every customer has a different ID, and it doesn't represent behavior or characteristics.
customer_ids = churn_dataset["CustomerID"].copy()
# Now remove the CustomerID column
churn_dataset = churn_dataset.drop("CustomerID",axis=1)
# Now verify
churn_dataset.columns


Seperate Feature And Scalaing

In [ ]:
X = churn_dataset.drop("ChurnStatus",axis=1)
y = churn_dataset["ChurnStatus"]
display(X.head())
display(y.head())

Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test,customer_train,customer_test = train_test_split(X,y,customer_ids,test_size=0.20,random_state=42)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Model Deployment

In [ ]:
from sklearn.linear_model import LogisticRegression

# Create Logistic Regression model
model = LogisticRegression(random_state=42)

# Train the model
model.fit(X_train, y_train)

# Predict class labels
y_pred = model.predict(X_test)

# Predict churn probabilities
y_prob = model.predict_proba(X_test)[:, 1]

print("Model Training Completed Successfully!")

**Model Evaluation**

Import Libraries

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)

import matplotlib.pyplot as plt
import seaborn as sns

Calculate Evaluation mertrices

In [ ]:
# Accuracy
accuracy = accuracy_score(y_test, y_pred)

# Precision
precision = precision_score(y_test, y_pred)

# Recall
recall = recall_score(y_test, y_pred)

# F1 Score
f1 = f1_score(y_test, y_pred)

# ROC-AUC Score
roc_auc = roc_auc_score(y_test, y_prob)

Result

In [ ]:
print("Accuracy Score :", accuracy)
print("Precision Score:", precision)
print("Recall Score   :", recall)
print("F1 Score       :", f1)
print("ROC-AUC Score  :", roc_auc)

Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

print(cm)

Plot the confusion Matrix

In [ ]:
plt.figure(figsize=(6,4))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")

plt.show()

ROC Curve

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(6,4))

plt.plot(fpr, tpr, label="ROC Curve")
plt.plot([0,1], [0,1], linestyle="--")

plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()

plt.show()

High Risk Customer Identification

In [ ]:
# DataFrame for results
results = pd.DataFrame({
    "CustomerID": customer_test.values,
    "Actual": y_test.values,
    "Predicted": y_pred,
    "Probability": y_prob
})

display(results.head())

# Risk Category
def risk_category(probability):
    if probability <= 0.40:
        return "Low Risk"
    elif probability <= 0.70:
        return "Medium Risk"
    else:
        return "High Risk"

# Apply Risk Category
results["Risk_Category"] = results["Probability"].apply(risk_category)

# Display Final Output
display(results.head())

# count customer risk in each group
print(results["Risk_Category"].value_counts())

# Saved Results
results.to_csv("results.csv",index=False)
print("Changes Saved Successfully")